# 실제 조사자료 기반 다점포 실험 v3

한국농촌경제연구원 2025 외식업체 경영실태조사: https://www.krei.re.kr/foodInfo/page/352?cmd=view&pageIndex=1&pst=505151

업종별 조사 가중치로 점포 수를 배분하고, 업종 안에서 방문자 경험분포의 확률구간을 최대 8분할해 실제 가게 한 행을 선택합니다. 8개씩 반복하고 나머지는 작은 블록으로 처리합니다. 모든 특성을 독립적으로 섞는 다변량 LHS는 아닙니다. 실제 관측 조합을 보존하지만 64점포의 상관계수와 업종 비율은 모집단과 정확히 같지 않습니다. 작은 업종이 배정되지 않거나 같은 가게가 중복 선택될 수 있으며 감사 기록에 남습니다.

대상: 일반 식사·간이음식·제과·커피·비알코올 음료점. 구내식당·출장음식·주점은 명시적으로 제외합니다. 방문자 0명과 큰 관측값은 보존합니다. 날짜별 방문 기록이 아니므로 일별 고객 수는 조사 평균으로 고정합니다. 실제 관측 메뉴가격은 천 원 단위로 변환하고 영업시간은 정수 시간으로 적용합니다. 좌석·객단가·인력·폐기량·비용·판매경로는 같은 행에서 보존하지만 모두 시뮬레이션 동작에 직접 쓰이는 것은 아닙니다.

조리인력 배분·메뉴 수·원가율·조리시간·폐기위험·재고·취향·고객 예산·대기 만족도·시간대별 도착은 아직 모형 가정입니다. 원가율은 실제 매장 비용을 개별 메뉴 비용으로 오인하지 않도록 기존 가정으로 남깁니다. 고객 예산도 실제 분포가 아니므로 고가 메뉴 가게의 결과는 주의해야 합니다. 만족도 안정화는 최적해나 실제 고객 효과의 증명이 아닙니다.

Google Drive에 가게별 중간 저장, 날짜별 모델 백업과 CSV를 누적합니다. 데이터나 코드가 달라지면 기존 실행과 섞지 않습니다. 한 실행 폴더에는 런타임 하나만 사용하세요. 체크포인트는 본인이 만든 파일만 열어야 합니다. 해시 검사는 손상을 감지하지만 악성 파일을 안전하게 만들지는 않습니다. 원자료는 공식 서버에서 직접 받으며 프로그램에 재배포하지 않습니다.


In [ ]:
import base64, io, zipfile, sys, subprocess
from pathlib import Path
ROOT = Path('/content/kiosk_empirical_v3')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:
    archive.extractall(ROOT)
sys.path.insert(0, str(ROOT))
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'openpyxl==3.1.5'])
print('준비 완료. 이전 버전을 실행한 런타임이라면 먼저 런타임을 다시 시작하세요.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, urllib.request
from kiosk_sim.empirical_stores import SOURCE_URL, SOURCE_SHA256
DATA = Path('/content/drive/MyDrive/Kiosk_LHS/source_2025.xlsx')
DATA.parent.mkdir(parents=True, exist_ok=True)
if not DATA.exists():
    temp = DATA.with_suffix('.download')
    try:
        urllib.request.urlretrieve(SOURCE_URL, temp)
        if hashlib.sha256(temp.read_bytes()).hexdigest() != SOURCE_SHA256:
            raise ValueError('공식 원자료와 해시가 다릅니다. 다른 파일을 대체 적용하지 않습니다.')
        temp.replace(DATA)
    except Exception:
        print('자동 다운로드 실패. 공식 엑셀을 위 DATA 경로에 올리고 이 셀을 다시 실행하세요.')
        raise
assert hashlib.sha256(DATA.read_bytes()).hexdigest() == SOURCE_SHA256, '원자료 버전 불일치'
from kiosk_sim.lhs_study import StudyConfig
CONFIG = StudyConfig(store_count=64, visitor_distribution='empirical_donors_2025',
                     survey_path=str(DATA), seed=20261006, lhs_strata=8,
                     min_days=20, max_days=160, stability_window=8,
                     tolerance=.003, required_stable_rounds=3)
RUN_NAME = 'empirical_v3_seed20261006'
SAVE_DIR = DATA.parent / RUN_NAME
print('백업 위치:', SAVE_DIR)


In [ ]:
import os, time
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from kiosk_sim.durable_study import DurableStudy
from kiosk_sim.lhs_study import MODEL_LABELS
run = DurableStudy(CONFIG, SAVE_DIR)
study = run.study
print('마지막 완료 날짜:', study.day, '종료 사유:', study.stop_reason)
print('표본 점검:', study.sampling_audit)
print('중복 선택은 가중 경험분포 표집의 일부입니다. 별개 고객 난수로 실행합니다.')

def plot_store(index=0):
    case = run.study.cases[index]
    fig, axes = plt.subplots(1, 3, figsize=(16,4))
    for ax, metric in zip(axes, ['score','customer_satisfaction','store_satisfaction']):
        for model in MODEL_LABELS:
            rows = [r for r in case.records if r['model'] == model]
            ax.plot([r['day'] for r in rows], [r[metric] for r in rows], label=model)
        ax.set(title=metric, xlabel='Day')
        ax.grid(alpha=.2)
    axes[0].legend(fontsize=7)
    fig.suptitle(f"Store {case.store_id} | {case.features['industry_label']} | visitors={case.features['visitor_mean']:g}")
    plt.tight_layout()
    plt.show()

stop_button = widgets.Button(description='오늘 완료 후 중지', button_style='warning')
stop_button.on_click(lambda _: (SAVE_DIR / 'STOP').touch())
picker = widgets.IntSlider(value=0, min=0, max=CONFIG.store_count-1, description='Store')
display(stop_button, picker)
print('버튼 응답이 늦으면 Drive 실행 폴더에 STOP 파일을 만드세요.')


## 실행 / 이어서 실행
일시 중단 후에는 설정 및 복구 셀부터 다시 실행하세요. 완료되지 않은 날짜의 가게별 저장도 복구됩니다. 강제 런타임 종료 후 RUNNING.lock이 남으면 이전 실행이 확실히 끝났는지 확인한 뒤 그 파일만 삭제하세요. 정상 종료 사유가 있는 실험은 자동 재시작하지 않습니다. 설정 변경은 새 RUN_NAME을 사용하세요.


In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
output = widgets.Output()
display(output)
workers = max(1, min(os.cpu_count() or 1, 4))
started = time.monotonic()
def progress(day, done, total):
    print(f'날짜 {day}: {done}/{total} 가게 저장 완료', end='\r')
background = ThreadPoolExecutor(max_workers=1)
try:
    while not run.study.finished():
        job = background.submit(run.next_day, workers=workers, progress=progress)
        await asyncio.shield(asyncio.wrap_future(job))
        study = run.study
        with output:
            clear_output(wait=True)
            print('완료 날짜:', study.day, '| 경과 분:', round((time.monotonic()-started)/60,2))
            print(study.analysis()['text'])
            plot_store(picker.value)
        if study.day % 5 == 0:
            study.export(desktop=SAVE_DIR)
except KeyboardInterrupt:
    print('중단했습니다. 완료된 가게의 저장은 유지됩니다. 복구 셀부터 다시 실행하세요.')
finally:
    # A UI interrupt cannot kill a Python thread safely; finish its current date.
    background.shutdown(wait=True)
    study = run.study
    print('정리된 결과:', study.export(desktop=SAVE_DIR))
    print('종료 사유:', study.stop_reason or '일시 중단 또는 오류; 저장된 위치에서 복구 가능')


In [ ]:
display(widgets.interactive(plot_store, index=widgets.IntSlider(value=0,min=0,max=CONFIG.store_count-1,description='Store')))
